## Setup & Dependencies
First, we'll install the necessary libraries and define our file paths.

In [1]:
# !pip install transformers torch networkx tqdm

In [2]:
import json
import re
import unicodedata
import os
from collections import Counter, defaultdict
from pathlib import Path
from typing import Optional

import networkx as nx
from tqdm.auto import tqdm
from transformers import pipeline

# Configuration
BOOKS_DIR  = Path("books") 
OUTPUT_DIR = Path("data")
CHAPTERS_CACHE = OUTPUT_DIR / "chapters_cache.json"

# ── Pipeline configuration ──────────────────────────────────────────────────
NER_MODEL_PIPELINE = "dslim/bert-large-NER"
MAX_TOKENS_P = 400   # max words per chunk
STRIDE_P     = 50    # overlap between chunks
COOC_WINDOW = 15 

# STOP_NAMES_P = {
#     "lord", "ser", "lady", "king", "queen", "prince", "princess",
#     "maester", "septon", "father", "mother", "brother", "sister",
#     "son", "daughter", "man", "woman", "boy", "girl", "old", "young",
#     "first", "second", "hand", "night", "watch", "wall", "iron",
#     "sept", "god", "gods",
# }
# NLTK's list of english stopwords
STOP_WORDS_P = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Chapter Extraction with Caching
This section handles the heavy lifting of reading the .txt files and splitting them by POV. I've added the logic to check if chapters_cache.json exists before running the regex.

In [3]:
CHAPTER_HEADING_RE = re.compile(r"^\s*([A-Z][A-Z\s]{1,30}?)\s*$", re.MULTILINE)

def get_chapters(force_refresh=False):
    """
    Loads chapters from cache if available, otherwise splits raw books and saves to cache.
    """
    if CHAPTERS_CACHE.exists() and not force_refresh:
        print(f"📦 Loading chapters from cache: {CHAPTERS_CACHE}")
        with open(CHAPTERS_CACHE, "r", encoding="utf-8") as f:
            return json.load(f)

    print("📖 Cache not found or refresh forced. Processing raw text files...")
    all_chapters = []
    paths = sorted(BOOKS_DIR.glob("*.txt"))
    
    if not paths:
        raise FileNotFoundError(f"No .txt files found in '{BOOKS_DIR}/'.")

    for path in paths:
        book_name = path.stem
        book_text = path.read_text(encoding="utf-8", errors="replace")
        matches = list(CHAPTER_HEADING_RE.finditer(book_text))
        
        if not matches:
            all_chapters.append({
                "book": book_name, "chapter_idx": 0, "heading": "UNKNOWN",
                "pov_char": "UNKNOWN", "text": book_text
            })
            continue

        for i, match in enumerate(matches):
            heading_raw = match.group(1).strip()
            body_start = match.end()
            body_end = matches[i + 1].start() if i + 1 < len(matches) else len(book_text)
            body = book_text[body_start:body_end].strip()

            if body:
                all_chapters.append({
                    "book": book_name,
                    "chapter_idx": i,
                    "heading": heading_raw,
                    "pov_char": heading_raw.title(),
                    "text": body,
                })

    # Save to cache
    with open(CHAPTERS_CACHE, "w", encoding="utf-8") as f:
        json.dump(all_chapters, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Saved {len(all_chapters)} chapters to cache.")
    return all_chapters

# Execution
chapters_data = get_chapters()

📖 Cache not found or refresh forced. Processing raw text files...
✅ Saved 340 chapters to cache.


## NER Engine & Text Processing
Here we initialize the BERT model and define the normalization functions.

In [4]:
# ── Helper functions ──────────────────────────────────
def normalise_name(raw: str) -> str:
    name = raw.replace("##", "")
    name = re.sub(r"\s+", " ", name)
    name = name.strip(" ,.'\"\u2014-")
    name = unicodedata.normalize("NFC", name)
    return name.title()

def is_valid_name(name: str) -> bool:
    name_lower = name.lower()
    if name_lower in STOP_WORDS_P: # or name_lower in STOP_NAMES_P:
        return False
    if len(name) < 2:
        return False
    if not any(c.isalpha() for c in name) or name.isdigit():
        return False
    return True

def chunk_text_p(text, max_words=MAX_TOKENS_P, stride=STRIDE_P):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words - stride):
        chunks.append(" ".join(words[i : i + max_words]))
        if i + max_words >= len(words):
            break
    return chunks

# ── Load model ───────────────────────────────────────────────────────────────
print(f"🤖 Loading NER model: {NER_MODEL_PIPELINE}")
ner_pipeline_base = pipeline(
    task="ner",
    model=NER_MODEL_PIPELINE,
    aggregation_strategy="simple",
    device=-1,   # set to 0 for GPU
)

print("✅ Model loaded.")

🤖 Loading NER model: dslim/bert-large-NER


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-large-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded.


## Relationship Network Logic
This builds the co-occurrence graph based on sentence proximity.

In [5]:
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

def build_network(chapter_text, persons, pov_char):
    G = nx.Graph()
    for char, count in persons.items():
        G.add_node(char, mentions=count, is_pov=(char == normalise_name(pov_char)))
    
    # Ensure POV is a node
    pov_name = normalise_name(pov_char)
    if is_valid_name(pov_name) and pov_name not in G:
        G.add_node(pov_name, mentions=0, is_pov=True)

    # Co-occurrence matching
    sentences = SENTENCE_SPLIT_RE.split(chapter_text)
    char_list = list(G.nodes)
    sentence_chars = []
    
    for sent in sentences:
        found = {name for name in char_list if name in sent}
        sentence_chars.append(found)

    for i in range(len(sentences)):
        window = set().union(*sentence_chars[i : i + COOC_WINDOW])
        window_chars = list(window)
        for idx_a in range(len(window_chars)):
            for idx_b in range(idx_a + 1, len(window_chars)):
                u, v = window_chars[idx_a], window_chars[idx_b]
                if G.has_edge(u, v): G[u][v]["weight"] += 1
                else: G.add_edge(u, v, weight=1)
    return G

## Run Pipeline & Export
The final execution loop. Note that since we loaded the chapters from the cache in Cell 2, this loop is now very clean.

In [6]:
all_chapters_networks = []
global_char_data = defaultdict(lambda: {"total_mentions": 0, "chapter_mentions": {}, "books": []})

for chapter in tqdm(chapters_data, desc="Building Networks"):
    ch_label = f"{chapter['book']}::{chapter['heading']}"
    
    # Run NER
    chapter_persons = Counter()
    chunks = chunk_text_p(chapter["text"])
    for chunk in chunks:
        ents = ner_pipeline_base(chunk)
        for ent in ents:
            if ent["entity_group"].startswith("PER"):
                name = normalise_name(ent["word"])
                if is_valid_name(name):
                    chapter_persons[name] += 1
    
    # Build Graph
    G = build_network(chapter["text"], chapter_persons, chapter["pov_char"])
    
    # Update Global Stats
    for char, count in chapter_persons.items():
        global_char_data[char]["total_mentions"] += count
        global_char_data[char]["chapter_mentions"][ch_label] = count
        if chapter["book"] not in global_char_data[char]["books"]:
            global_char_data[char]["books"].append(chapter["book"])
            
    # Save chapter record
    all_chapters_networks.append({
        "book": chapter["book"],
        "chapter_idx": chapter["chapter_idx"],
        "heading": chapter["heading"],
        "pov_char": chapter["pov_char"],
        "network": {
            "nodes": [{"id": n, **d} for n, d in G.nodes(data=True)],
            "edges": [{"source": u, "target": v, **d} for u, v, d in G.edges(data=True)]
        }
    })

# Final Export
with open(OUTPUT_DIR / "characters_books.json", "w") as f:
    json.dump(global_char_data, f, indent=2)
with open(OUTPUT_DIR / "networks_v0.json", "w") as f:
    json.dump(all_chapters_networks, f, indent=2)

print("✨ Analysis Complete!")

Building Networks:   0%|          | 0/340 [00:00<?, ?it/s]

✨ Analysis Complete!


## Export Networks as GEXF
This cell iterates through your networks.json and creates a standalone .gexf file for every chapter.

In [7]:
# Configuration for GEXF Export
GEXF_DIR = OUTPUT_DIR / "gexf_files_per_chapter"
GEXF_DIR.mkdir(exist_ok=True)

def export_networks_to_gexf_fixed(networks_file, fixed_weight=1.0):
    """
    Exports each chapter as a .gexf file where all edges have the 
    exact same weight, ensuring slim and uniform lines in Gephi.
    """
    with open(networks_file, "r", encoding="utf-8") as f:
        chapters = json.load(f)

    print(f"🕸️ Exporting uniform-edge GEXF files to {GEXF_DIR}...")

    for ch in tqdm(chapters, desc="Exporting Chapters"):
        net = ch["network"]
        book = ch["book"]
        heading = ch["heading"]
        idx = ch["chapter_idx"]
        
        G = nx.Graph()
        
        # Add Nodes
        for node in net["nodes"]:
            G.add_node(
                node["id"], 
                label=node["id"],
                mentions=node["mentions"], 
                is_pov=node.get("is_pov", False)
            )
        
        # Add Edges with a CONSTANT weight
        for edge in net["edges"]:
            # We ignore edge["weight"] from the JSON and use the fixed_weight instead
            G.add_edge(
                edge["source"], 
                edge["target"], 
                weight=fixed_weight
            )

        filename = f"{book}_{idx:02d}_{heading.replace(' ', '_')}.gexf"
        nx.write_gexf(G, GEXF_DIR / filename)

export_networks_to_gexf_fixed(OUTPUT_DIR / "networks_v0.json")

🕸️ Exporting uniform-edge GEXF files to data\gexf_files_per_chapter...


Exporting Chapters:   0%|          | 0/340 [00:00<?, ?it/s]